In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets huggingface_hub

In [ ]:
import os
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
df = pd.read_csv('cuidados_plantas.csv')
df.head()

In [ ]:
def row_to_text(row):
    return f"""### Instrucción:
Dame información sobre el cuidado de la planta {row['name']}.

### Respuesta:
La planta {row['name']} (nombre científico: {row['scientific_name']}) pertenece a la especie {row['species_id']}.
- Luz solar: {row['sunlight']}
- Riego: {row['watering']}
- Poda: {row['pruning']}"""

df['text'] = df.apply(row_to_text, axis=1)

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[['text']])
dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_train = dataset["train"].map(tokenize, batched=True, remove_columns=["text"])
tokenized_eval = dataset["test"].map(tokenize, batched=True, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir="./plantas-llm",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
repo_name = "plantas-llm-finetuned"

trainer.model.push_to_hub(repo_name, token=HF_TOKEN)
tokenizer.push_to_hub(repo_name, token=HF_TOKEN)

print(f"Modelo subido a: https://huggingface.co/{HfApi().whoami(token=HF_TOKEN)['name']}/{repo_name}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
from huggingface_hub import HfApi

username = HfApi().whoami(token=HF_TOKEN)['name']
repo_id = f"{username}/plantas-llm-finetuned"

tokenizer = AutoTokenizer.from_pretrained(repo_id, token=HF_TOKEN)
model_inf = AutoModelForCausalLM.from_pretrained(
    repo_id,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

In [ ]:
def preguntar(planta):
    prompt = f"""### Instrucción:
Dame información sobre el cuidado de la planta {planta}.

### Respuesta:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model_inf.device)

    with torch.no_grad():
        outputs = model_inf.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    respuesta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(respuesta.split("### Respuesta:")[-1].strip())

preguntar("Citrus berry")